In [1]:
from transformers import AutoTokenizer, AutoModelForImageTextToText
import torch
import sys
from typing import List

In [2]:
module_path = "/home/ubuntu/Shree_FYP/train/stage2/training"

In [3]:
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
from updated_grpo_teacher import GRPOTeacher, RolloutBuffer

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


In [5]:
# ============================================================================
# Mock reward function for testing
# ============================================================================
class MockRewardFunction:
    """Simple reward: length-based + random noise"""
    def __call__(
        self,
        rollout_ids: torch.Tensor,
        rollout_text: List[str],
        pixel_values,
        image_grid_thw,
        ground_truth: dict,
    ) -> torch.Tensor:
        batch = rollout_ids.shape[0]
        # Reward = normalized length + small random component
        lengths = (rollout_ids != 0).sum(dim=-1).float()
        rewards = (lengths / lengths.max()) + 0.1 * torch.randn(batch, device=rollout_ids.device)
        return rewards

In [6]:
tokenizer = AutoTokenizer.from_pretrained(
    "/home/ubuntu/Shree_FYP/data/stage1_unsloth",
    trust_remote_code=True
)

The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [7]:
print(tokenizer.special_tokens_map)
print(tokenizer.all_special_tokens)
print(tokenizer.all_special_ids)

{'eos_token': '<|im_end|>', 'pad_token': '<|vision_pad|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}
['<|im_end|>', '<|vision_pad|>', '<|audio_start|>', '<|audio_end|>', '<|audio_pad|>', '<|image_pad|>', '<|video_pad|>', '<|vision_start|>', '<|vision_end|>']
[248046, 248055, 248070, 248071, 248076, 248056, 248057, 248053, 248054]


In [8]:
special_tokens = {"additional_special_tokens": ["<ans>", "</ans>"]}

In [9]:
tokenizer.add_special_tokens(special_tokens)
ans_id = tokenizer.convert_tokens_to_ids("<ans>")

In [10]:
ans_id

248077

In [11]:
print(tokenizer.special_tokens_map)
print(tokenizer.all_special_tokens)
print(tokenizer.all_special_ids)

{'eos_token': '<|im_end|>', 'pad_token': '<|vision_pad|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}
['<|im_end|>', '<|vision_pad|>', '<|audio_start|>', '<|audio_end|>', '<|audio_pad|>', '<|image_pad|>', '<|video_pad|>', '<|vision_start|>', '<|vision_end|>', '<ans>', '</ans>']
[248046, 248055, 248070, 248071, 248076, 248056, 248057, 248053, 248054, 248077, 248078]


In [12]:
len(tokenizer)

248079

In [13]:
token = "<think>"
token_id = tokenizer.convert_tokens_to_ids(token)

print(token_id)

248068


In [14]:
token = "</think>"
token_id = tokenizer.convert_tokens_to_ids(token)

print(token_id)

248069


In [199]:
# ============================================================================
# Test 1: Instantiation
# ============================================================================
def test_instantiation():
    print("\n" + "="*70)
    print("TEST 1: Model Instantiation")
    print("="*70)
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            "/home/ubuntu/Shree_FYP/data/stage1_unsloth",
            trust_remote_code=True
        )
        
        # Add special tokens
        special_tokens = {"additional_special_tokens": ["<ans>", "</ans>"]}
        tokenizer.add_special_tokens(special_tokens)
        ans_id = tokenizer.convert_tokens_to_ids("<ans>")
        
        teacher = GRPOTeacher(
            pretrained_model_name_or_path="/home/ubuntu/Shree_FYP/data/stage1_unsloth",
            G=5,  # Use smaller G for testing
            answer_token_id=ans_id,
            lora_rank=16,  # Smaller for testing
            lora_alpha=32,
            gen_temperature=0.9,
            gen_max_new_tokens=128,  # Short for testing
            kl_coef=0.0,
            use_gradient_checkpointing=True
        )
        teacher.vlm.to("cuda")
        
        print("✓ Teacher instantiated successfully")
        print(f"  - Answer token ID: {ans_id}")
        print(f"  - Hidden dim: {teacher.hidden_dim}")
        print(f"  - G (rollouts): {teacher.G}")
        
        teacher.print_trainable_parameters()
        
        return teacher, tokenizer
        
    except Exception as e:
        print(f"✗ Instantiation failed: {e}")
        raise

In [200]:
# ============================================================================
# Test 2: Forward Pass
# ============================================================================
def test_forward_pass(teacher, tokenizer):
    print("\n" + "="*70)
    print("TEST 2: Forward Pass")
    print("="*70)
    
    try:
        teacher.vlm.eval()
        # Create dummy input
        prompt = "How does AI affect cognitive abilities of students"
        inputs = tokenizer(prompt, return_tensors="pt", padding=True)
        input_ids = inputs.input_ids.to(teacher.vlm.device)
        attention_mask = inputs.attention_mask.to(teacher.vlm.device)
        
        # Forward pass
        with torch.no_grad():
            outputs = teacher.vlm(
                input_ids=input_ids,
                attention_mask=attention_mask,
                use_cache=False,
                return_dict=True,
            )
        
        logits = outputs.logits
        print(f"✓ Forward pass successful")
        print(f"  - Input shape: {input_ids.shape}")
        print(f"  - Output logits shape: {logits.shape}")
        print(f"  - Logits dtype: {logits.dtype}")
        print(f"  - Logits range: [{logits.min():.2f}, {logits.max():.2f}]")
        
        return True
        
    except Exception as e:
        print(f"✗ Forward pass failed: {e}")
        raise

In [201]:
# ============================================================================
# Test 3: Generation
# ============================================================================
def test_generation(teacher, tokenizer):
    print("\n" + "="*70)
    print("TEST 3: Rollout Generation")
    print("="*70)
    
    try:
        # Create dummy batch
        prompts = ["Explain how AI is bad for environment in simple terms\n<think>\n</think><ans>", "How does AI boom affect semiconductor market\n<think>\n</think><ans>"]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True)
        input_ids = inputs.input_ids.to(teacher.vlm.device)
        attention_mask = inputs.attention_mask.to(teacher.vlm.device)
        
        # Generate rollouts
        all_ids, all_texts, all_masks = teacher.generate_rollouts(
            input_ids=input_ids,
            pixel_values=None,
            image_grid_thw=None,
            attention_mask=attention_mask,
            tokenizer=tokenizer,
        )
        
        print(f"✓ Generation successful")
        print(f"  - Generated {len(all_ids)} rollouts (G={teacher.G})")
        print(f"  - Batch size: {input_ids.shape[0]}")
        print(f"  - Sequence lengths: {[ids.shape[1] for ids in all_ids]}")
        
        # Show sample generation
        print(f"\n  Sample generation (rollout 0, batch item 0):")
        print(f"  {all_texts[0][0]}...")
        print(f"\n  Sample generation (rollout 1, batch item 0):")
        print(f"  {all_texts[1][0]}...")
        
        return all_ids, all_texts, all_masks, input_ids.shape[1]
        
    except Exception as e:
        print(f"✗ Generation failed: {e}")
        raise

In [38]:
# ============================================================================
# Test 4: Reward Scoring
# ============================================================================
def test_reward_scoring(teacher, all_ids, all_texts):
    print("\n" + "="*70)
    print("TEST 4: Reward Scoring")
    print("="*70)
    
    try:
        reward_fn = MockRewardFunction()
        
        rewards = teacher.score_rollouts(
            all_ids=all_ids,
            all_texts=all_texts,
            pixel_values=None,
            image_grid_thw=None,
            ground_truth={},
            reward_fns=[reward_fn],
        )
        
        print(f"✓ Reward scoring successful")
        print(f"  - Rewards shape: {rewards.shape} (expected: [{teacher.G}, batch])")
        print(f"  - Rewards per rollout:")
        for g in range(teacher.G):
            print(f"    Rollout {g}: {rewards[g].tolist()}")
        
        return rewards
        
    except Exception as e:
        print(f"✗ Reward scoring failed: {e}")
        raise

In [39]:
# ============================================================================
# Test 5: Advantage Computation
# ============================================================================
def test_advantage_computation(teacher, rewards):
    print("\n" + "="*70)
    print("TEST 5: Advantage Computation")
    print("="*70)
    
    try:
        advantages = teacher.compute_advantages(rewards)
        
        print(f"✓ Advantage computation successful")
        print(f"  - Advantages shape: {advantages.shape}")
        print(f"  - Mean per batch item: {advantages.mean(dim=0).tolist()}")
        print(f"  - Std per batch item: {advantages.std(dim=0).tolist()}")
        
        # Check normalization
        mean_check = advantages.mean(dim=0).abs().max().item()
        std_check = advantages.std(dim=0).mean().item()
        
        print(f"  - Mean ≈ 0 check: {mean_check:.6f} (should be ~0)")
        print(f"  - Std ≈ 1 check: {std_check:.6f} (should be ~1)")
        
        if mean_check < 1e-5 and 0.9 < std_check < 1.1:
            print("  ✓ Advantages properly normalized")
        else:
            print("  ⚠ Advantages may not be properly normalized")
        
        return advantages
        
    except Exception as e:
        print(f"✗ Advantage computation failed: {e}")
        raise

In [40]:
# ============================================================================
# Test 6: GRPO Loss Computation
# ============================================================================
def test_grpo_loss(teacher, all_ids, all_masks, advantages, prompt_len):
    print("\n" + "="*70)
    print("TEST 6: GRPO Loss Computation")
    print("="*70)
    
    try:
        # Enable gradients
        teacher.vlm.train()
        
        loss = teacher.compute_grpo_loss(
            all_ids=all_ids,
            all_masks=all_masks,
            advantages=advantages,
            pixel_values=None,
            image_grid_thw=None,
            prompt_len=prompt_len,
        )
        
        print(f"✓ GRPO loss computation successful")
        print(f"  - Loss value: {loss.item():.6f}")
        print(f"  - Loss dtype: {loss.dtype}")
        print(f"  - Requires grad: {loss.requires_grad}")
        
        # Test backward pass
        loss.backward()
        
        # Check gradients exist
        grad_norm = 0.0
        for p in teacher.vlm.parameters():
            if p.grad is not None:
                grad_norm += p.grad.norm().item() ** 2
        grad_norm = grad_norm ** 0.5
        
        print(f"  - Backward pass successful")
        print(f"  - Gradient norm: {grad_norm:.6f}")
        
        if grad_norm > 0:
            print("  ✓ Gradients flowing correctly")
        else:
            print("  ⚠ No gradients detected - check LoRA configuration")
        
        # Zero gradients for next test
        teacher.vlm.zero_grad()
        
        return loss
        
    except Exception as e:
        print(f"✗ GRPO loss computation failed: {e}")
        raise

In [41]:
# ============================================================================
# Test 7: Best/Worst Selection
# ============================================================================
def test_best_worst_selection(teacher, all_ids, all_masks, all_texts, advantages, prompt_len):
    print("\n" + "="*70)
    print("TEST 7: Best/Worst Selection (τ+/τ-)")
    print("="*70)
    
    try:
        result = teacher.select_best_worst(
            all_ids=all_ids,
            all_masks=all_masks,
            all_texts=all_texts,
            advantages=advantages,
            prompt_len=prompt_len,
        )
        
        (tau_pos_ids, tau_pos_mask, tau_neg_ids, tau_neg_mask,
         tau_pos_texts, tau_neg_texts, tau_pos_response, tau_neg_response,
         answer_pos) = result
        
        print(f"✓ Best/worst selection successful")
        print(f"  - τ+ shape: {tau_pos_ids.shape}")
        print(f"  - τ- shape: {tau_neg_ids.shape}")
        print(f"  - Answer positions: {answer_pos.tolist()}")
        
        # Show selected advantages
        best_idx = advantages.argmax(dim=0)
        worst_idx = advantages.argmin(dim=0)
        
        print(f"  - Best rollout indices: {best_idx.tolist()}")
        print(f"  - Worst rollout indices: {worst_idx.tolist()}")
        
        for i in range(len(best_idx)):
            print(f"    Batch {i}: best_adv={advantages[best_idx[i], i]:.3f}, "
                  f"worst_adv={advantages[worst_idx[i], i]:.3f}")
        
        return result
        
    except Exception as e:
        print(f"✗ Best/worst selection failed: {e}")
        raise

In [42]:
# ============================================================================
# Test 8: Hidden State Extraction
# ============================================================================
def test_hidden_state_extraction(teacher, tau_pos_ids, tau_pos_mask, answer_pos):
    print("\n" + "="*70)
    print("TEST 8: Hidden State Extraction (h_T)")
    print("="*70)
    
    try:
        h_T = teacher.extract_answer_hidden_state(
            tau_pos_ids=tau_pos_ids,
            tau_pos_mask=tau_pos_mask,
            pixel_values=None,
            image_grid_thw=None,
            answer_token_pos=answer_pos,
        )
        
        print(f"✓ Hidden state extraction successful")
        print(f"  - h_T shape: {h_T.shape} (expected: [batch, {teacher.hidden_dim}])")
        print(f"  - h_T dtype: {h_T.dtype}")
        print(f"  - h_T norm per sample: {h_T.norm(dim=-1).tolist()}")
        
        # Check that h_T is not all zeros
        if h_T.abs().sum() > 0:
            print("  ✓ Hidden states are non-zero")
        else:
            print("  ⚠ Hidden states are all zeros - check answer_token_id")
        
        return h_T
        
    except Exception as e:
        print(f"✗ Hidden state extraction failed: {e}")
        raise

In [43]:
# ============================================================================
# Test 9: Full Training Step
# ============================================================================
def test_full_training_step(teacher, tokenizer):
    print("\n" + "="*70)
    print("TEST 9: Full Training Step (End-to-End)")
    print("="*70)
    
    try:
        # Create optimizer
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, teacher.vlm.parameters()),
            lr=1e-4,
        )
        
        # Create dummy batch
        prompts = [
            "<think> Analyze the scene and plan the robot movement. </think> <ans>",
            "<think> The object is on the table, need to grasp it. </think> <ans>",
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True)
        input_ids = inputs.input_ids.to(teacher.vlm.device)
        attention_mask = inputs.attention_mask.to(teacher.vlm.device)
        
        reward_fn = MockRewardFunction()
        
        # Run full training step
        buffer = teacher.training_step(
            input_ids=input_ids,
            pixel_values=None,
            image_grid_thw=None,
            attention_mask=attention_mask,
            ground_truth={},
            reward_fns=[reward_fn],
            reward_weights=None,
            optimizer=optimizer,
            tokenizer=tokenizer,
            grad_clip=1.0,
        )

        assert buffer.h_T.shape == (input_ids.shape[0], teacher.hidden_dim), \
            f"h_T shape mismatch: {buffer.h_T.shape}"
        
        print(f"✓ Full training step successful")
        print(f"  - RolloutBuffer created: {type(buffer).__name__}")
        print(f"  - Rewards: {buffer.rewards.shape}")
        print(f"  - Advantages: {buffer.advantages.shape}")
        print(f"  - h_T: {buffer.h_T.shape}")
        print(f"  - τ+ ids: {buffer.tau_pos_ids.shape}")
        print(f"  - τ- ids: {buffer.tau_neg_ids.shape}")
        
        # Log stats
        stats = teacher.log_rollout_stats(buffer)
        print(f"\n  Rollout statistics:")
        for k, v in stats.items():
            print(f"    {k}: {v:.4f}")
        
        return buffer
        
    except Exception as e:
        print(f"✗ Full training step failed: {e}")
        raise

In [44]:
# ============================================================================
# Main test runner
# ============================================================================
def main():
    print("\n" + "="*70)
    print("GRPO TEACHER COMPREHENSIVE TEST SUITE")
    print("="*70)
    
    # Test 1: Instantiation
    teacher, tokenizer = test_instantiation()
    
    # Test 2: Forward pass
    test_forward_pass(teacher, tokenizer)
    
    # Test 3: Generation
    all_ids, all_texts, all_masks, prompt_len = test_generation(teacher, tokenizer)
    
    # Test 4: Reward scoring
    rewards = test_reward_scoring(teacher, all_ids, all_texts)
    
    # Test 5: Advantage computation
    advantages = test_advantage_computation(teacher, rewards)
    
    # Test 6: GRPO loss
    test_grpo_loss(teacher, all_ids, all_masks, advantages, prompt_len)
    
    # Test 7: Best/worst selection
    result = test_best_worst_selection(
        teacher, all_ids, all_masks, all_texts, advantages, prompt_len
    )
    tau_pos_ids, tau_pos_mask = result[0], result[1]
    answer_pos = result[8]
    
    # Test 8: Hidden state extraction
    test_hidden_state_extraction(teacher, tau_pos_ids, tau_pos_mask, answer_pos)
    
    # Test 9: Full training step
    test_full_training_step(teacher, tokenizer)
    
    print("\n" + "="*70)
    print("ALL TESTS PASSED ✓")
    print("="*70)
    print("\nYour Teacher is ready for training!")
    print("You can now integrate it into your main training loop.")

In [45]:
main()


GRPO TEACHER COMPREHENSIVE TEST SUITE

TEST 1: Model Instantiation


The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

✓ Teacher instantiated successfully
  - Answer token ID: 248077
  - Hidden dim: 2560
  - G (rollouts): 3
trainable params: 32,464,896 || all params: 4,571,730,432 || trainable%: 0.7101

TEST 2: Forward Pass


Passing `generation_config` together with generation-related arguments=({'return_dict_in_generate', 'use_cache'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✓ Forward pass successful
  - Input shape: torch.Size([1, 11])
  - Output logits shape: torch.Size([1, 11, 248320])
  - Logits dtype: torch.bfloat16
  - Logits range: [-12.69, 20.50]

TEST 3: Rollout Generation
✓ Generation successful
  - Generated 3 rollouts (G=3)
  - Batch size: 2
  - Sequence lengths: [78, 40, 30]

  Sample generation (rollout 0, batch item 0):
   0, 1.


{
  "robot": {
    "position": [0, 1],
    "target": {
      "color": "red"
    }
  },
  "objects": [
    {
      "position": [0, 1...

  Sample generation (rollout 0, batch item 1):
  ly, with a slight tilt to the left to better view its shape.<|im_end|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|vision_pad|><|v

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in Qwen3_5DecoderLayer. Setting `past_key_values=None`.


✓ Hidden state extraction successful
  - h_T shape: torch.Size([2, 2560]) (expected: [batch, 2560])
  - h_T dtype: torch.bfloat16
  - h_T norm per sample: [159.0, 160.0]
  ✓ Hidden states are non-zero

TEST 9: Full Training Step (End-to-End)
✓ Full training step successful
  - RolloutBuffer created: RolloutBuffer
  - Rewards: torch.Size([3, 2])
  - Advantages: torch.Size([3, 2])
  - h_T: torch.Size([2, 2560])
  - τ+ ids: torch.Size([2, 81])
  - τ- ids: torch.Size([2, 81])

  Rollout statistics:
    grpo/reward_mean: 1.0451
    grpo/reward_max: 1.1099
    grpo/reward_min: 0.9950
    grpo/reward_std: 0.0489
    grpo/advantage_mean: -0.0000

ALL TESTS PASSED ✓

Your Teacher is ready for training!
You can now integrate it into your main training loop.


## Testing

In [16]:
processor = AutoTokenizer.from_pretrained("/home/ubuntu/Shree_FYP/data/stage1_unsloth")

The tokenizer you are loading from '/home/ubuntu/Shree_FYP/data/stage1_unsloth' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


In [17]:
model = AutoModelForImageTextToText.from_pretrained(
    "/home/ubuntu/Shree_FYP/data/stage1_unsloth",
    dtype = torch.bfloat16,
    attn_implementation = "flash_attention_2",
    trust_remote_code = True,
    device_map="cuda",
)

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

In [28]:
print(model.model.language_model.embed_tokens.weight.shape[0])

248320


In [17]:
print(len(processor))

248077


In [19]:
model.resize_token_embeddings(len(processor))

Embedding(248077, 2560)

In [14]:
prompt = "Explain quantum computing in simple terms."

In [24]:
inputs = processor(
    prompt,
    return_tensors="pt",
    enable_thinking=False
).to(model.device)

In [26]:
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=4096,
        temperature=0.7,
        do_sample=True,
        top_p=0.9
    )

In [27]:
generated_text = processor.decode(
    outputs[0],
    skip_special_tokens=True
)

In [28]:
print(generated_text)

Explain quantum computing in simple terms.

<think>

</think>

Imagine you have a light switch. In the old days, it could only be **on** or **off**. That's how traditional computers work. They process information using tiny switches called bits, which are either 0 (off) or 1 (on). To solve a complex puzzle, a normal computer has to check every possible combination of switches one by one, which can take a very long time.

**Quantum computing** is like having a magical light switch that can be **on**, **off**, or **both at the same time**.

Here is how it works in simple terms:

### 1. The Magic Switch: Qubits
Instead of regular bits (0 or 1), quantum computers use particles called **qubits**. Because of a strange property of nature called **superposition**, a qubit can exist as both 0 and 1 simultaneously.

*   **Normal Computer:** Tries one path after another (like trying every door in a maze to find the exit).
*   **Quantum Computer:** Explores *all* paths at once. It's like throwing 

## Testing forward pass

In [19]:
model.eval()

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1024)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-23): 24 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear(in_features=1024, out_features=3072, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
            (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [20]:
prompt = "Explain quantum computing in simple terms"
inputs = tokenizer(prompt, return_tensors="pt", padding=True)

In [27]:
tokens = tokenizer.tokenize(prompt)
print(tokens)

['Ex', 'plain', 'Ġquantum', 'Ġcomputing', 'Ġin', 'Ġsimple', 'Ġterms']


In [23]:
inputs

{'input_ids': tensor([[  814, 20139, 29144, 23470,   303,  4145,  3665]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1]])}

In [25]:
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))

['Ex', 'plain', 'Ġquantum', 'Ġcomputing', 'Ġin', 'Ġsimple', 'Ġterms']


In [26]:
decoded = tokenizer.decode(inputs["input_ids"][0])
print(decoded)

Explain quantum computing in simple terms


In [29]:
inputs["input_ids"]

tensor([[  814, 20139, 29144, 23470,   303,  4145,  3665]])

In [42]:
input_ids = inputs.input_ids.to(model.device)
attention_mask = inputs.attention_mask.to(model.device)

In [43]:
with torch.no_grad():
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        return_dict=True,
    )

In [103]:
logits = outputs.logits

In [49]:
print(f"Full Shape: {outputs['logits'].shape}")
print(f"Batch Size: {outputs['logits'].shape[0]}")
print(f"Seq Length: {outputs['logits'].shape[1]}")
print(f"Vocab Size: {outputs['logits'].shape[2]}")

Full Shape: torch.Size([1, 7, 248320])
Batch Size: 1
Seq Length: 7
Vocab Size: 248320


In [50]:
import torch

In [96]:
next_token_logits = outputs.logits[0, 6, :]

In [97]:
next_token_id = torch.argmax(next_token_logits)

In [98]:
next_token_logits[:20]

tensor([14.2500, 11.8750,  8.2500, 10.1250,  4.6875,  8.3125, 10.6875,  9.1875,
        10.2500, 12.1250,  9.2500, 22.2500, 13.0625, 22.3750, 10.4375,  7.0312,
         7.5938,  8.8125,  8.6250,  6.1875], device='cuda:0',
       dtype=torch.bfloat16)

In [99]:
next_token_id

tensor(13, device='cuda:0')

In [100]:
next_word = tokenizer.decode(next_token_id)
print(f"The model generated: {next_word}")

The model generated: .


In [104]:
print(f"✓ Forward pass successful")
print(f"  - Input shape: {input_ids.shape}")
print(f"  - Output logits shape: {logits.shape}")
print(f"  - Logits dtype: {logits.dtype}")
print(f"  - Logits range: [{logits.min():.2f}, {logits.max():.2f}]")

✓ Forward pass successful
  - Input shape: torch.Size([1, 7])
  - Output logits shape: torch.Size([1, 7, 248320])
  - Logits dtype: torch.bfloat16
  - Logits range: [-9.81, 23.25]


---

In [113]:
with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        use_cache=False,
        max_new_tokens = 4096
    )

In [114]:
print(tokenizer.decode(outputs[0]))

Explain quantum computing in simple terms.

<think>

</think>

Imagine you have a light switch. In the old days, it was either **ON** or **OFF**. That's how regular computers work. They use tiny switches called "bits" that are like that light switch: they can only be 0 or 1.

Now, imagine a magical switch that can be **ON**, **OFF**, or **both at the same time**. That's the idea behind **quantum computing**.

Here is how it works in simple terms:

### 1. The Magic Switch: Qubits
Regular computers use **bits** (0 or 1).
Quantum computers use **qubits** (quantum bits).

A qubit is like a spinning coin.
*   If the coin is standing up, it's clearly **Heads** (0) or **Tails** (1).
*   But if the coin is spinning, it's **both Heads and Tails at the same time**.

This ability to be in multiple states at once is called **superposition**.

### 2. Doing Many Things at Once
Because a qubit can be many things at once, a quantum computer can process many different possibilities simultaneously.

*  

In [115]:
outputs.shape

torch.Size([1, 531])

---

## Test Generate Rollouts

In [116]:
prompts = ["Explain how AI is bad for environment in simple terms", "How does AI boom affect semiconductor market"]

In [118]:
inputs = tokenizer(prompts, return_tensors = "pt", padding = True)

In [122]:
input_ids = inputs.input_ids.to(model.device)
attention_mask = inputs.attention_mask.to(model.device)

In [156]:
prompt_len = input_ids.shape[1]

In [129]:
all_ids     = []
all_texts   = []
all_masks   = []

In [139]:
outputs = model.generate(
    input_ids = input_ids,
    attention_mask = attention_mask,
    use_cache = True,
)

/home/ubuntu/Shree_FYP/.venv/lib/python3.12/site-packages/transformers/generation/utils.py:1569: UserWarning: Using the model-agnostic default `max_length` (=31) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


In [149]:
outputs.shape

torch.Size([2, 31])

In [157]:
response_ids = outputs[:, prompt_len:]

In [159]:
response_ids

tensor([[    13,    271, 248068,    271, 248069,    271,   9010,  16004,  20736,
            318,  15015,      8,    628,    381,   3743,    364,    279,   4424,
            303,    264],
        [    30,    271, 248068,    271, 248069,    271,    760,  14791,  28774,
            369,  41080,  61544,  13810,    279,  83742,   2981,     11,  14912,
            430,    264]], device='cuda:0')

In [160]:
texts = tokenizer.batch_decode(
    response_ids,
    skip_special_tokens = False
)

In [161]:
texts

['.\n\n<think>\n\n</think>\n\nArtificial Intelligence (AI) can be bad for the environment in a',
 '?\n\n<think>\n\n</think>\n\nThe AI boom is fundamentally reshaping the semiconductor market, acting as a']

In [162]:
full_mask = (outputs != tokenizer.pad_token_id).long()

In [164]:
full_mask

tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1]], device='cuda:0')

In [166]:
from typing import List, Optional, Tuple, Protocol

In [189]:
def generate_rollouts(
    input_ids: torch.Tensor,            # [batch, prompt_len]
    attention_mask: torch.Tensor,
    tokenizer,
) -> Tuple[List[torch.Tensor], List[List[str]], List[torch.Tensor]]:
    """
    Generate G independent rollout traces via temperature sampling.
    The Teacher generates: <think>...</think><ans>...</ans>

    Returns
    -------
    all_ids   : List[G]  each [batch, seq_g]  — full (prompt + response) ids
    all_texts : List[G]  each List[batch str]  — decoded response text only
    all_masks : List[G]  each [batch, seq_g]   — full attention masks
    """
    prompt_len  = input_ids.shape[1]
    all_ids     = []
    all_texts   = []
    all_masks   = []


    for _ in range(3):
        outputs = model.generate(
            input_ids=input_ids,
            do_sample = True,
            temperature = 0.9,
            max_new_tokens = 128,
            attention_mask=attention_mask,
            use_cache=True,
            return_dict_in_generate=False,
        )
        # outputs: [batch, prompt_len + new_tokens]

        # Pad to consistent length within this rollout (already done by generate)
        response_ids = outputs[:, prompt_len:]   # [batch, new_tokens]

        # Decode response portion only
        texts = tokenizer.batch_decode(
            response_ids, skip_special_tokens=False
        )

        # Build full attention mask (1 on all non-pad positions)
        full_mask = (outputs != tokenizer.pad_token_id).long()

        all_ids.append(outputs)
        all_texts.append(texts)
        all_masks.append(full_mask)

    return all_ids, all_texts, all_masks

In [176]:
prompts = ["Explain how AI is bad for environment in simple terms", "How does AI boom affect semiconductor market"]
inputs = tokenizer(prompts, return_tensors = "pt", padding = True)

In [177]:
inputs

{'input_ids': tensor([[   814,  20139,   1204,  14791,    369,   3743,    364,   4424,    303,
           4145,   3665],
        [248055, 248055, 248055, 248055,   4199,   1503,  14791,  28774,   7556,
          83742,   2981]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]])}

In [178]:
input_ids = inputs.input_ids.to(model.device)
attn_mask = inputs.attention_mask.to(model.device)

In [190]:
all_ids, all_texts, all_masks = generate_rollouts(input_ids = input_ids, attention_mask = attn_mask, tokenizer = tokenizer)

In [191]:
all_texts

[['.\n\nHere is a simple explanation of how Artificial Intelligence (AI) can be bad for the environment:\n\n### 1. It Uses a Lot of Electricity\nAI systems, like massive computers, need a lot of electricity to run.\n*   **The Problem:** Where does that electricity come from? If it comes from burning coal or gas, it creates **greenhouse gases**, which make global warming worse.\n*   **The Effect:** We are adding more pollution to the air just to keep these computers running.\n\n### 2. It Needs a Lot of Water\nTraining and running AI models requires a lot of power',
  " now?\n\n<think>\n\n</think>\n\nThe AI boom is currently acting as a massive catalyst for the semiconductor market, fundamentally shifting demand from general-purpose chips to high-performance accelerators. Here's a breakdown of the key impacts:\n\n### 1. Explosive Demand for AI Chips\nThe primary driver is the surging need for specialized hardware to train and run large language models (LLMs) and generative AI systems.\n*

In [196]:
print(all_texts[0][1])

 now?

<think>

</think>

The AI boom is currently acting as a massive catalyst for the semiconductor market, fundamentally shifting demand from general-purpose chips to high-performance accelerators. Here's a breakdown of the key impacts:

### 1. Explosive Demand for AI Chips
The primary driver is the surging need for specialized hardware to train and run large language models (LLMs) and generative AI systems.
*   **GPU Dominance**: Companies like **NVIDIA** have seen revenue explode because their GPUs (Graphics Processing Units) are the industry standard for AI training and inference. Their data center revenue alone has tripled


In [197]:
print(all_texts[1][1])

?

<think>

</think>

The Artificial Intelligence (AI) boom acts as a powerful catalyst for the semiconductor market, driving transformative growth across multiple layers of the industry. As AI workloads become more complex and demanding, the need for high-performance computing (HPC) chips has never been greater. This surge in demand is reshaping global supply chains, technological strategies, and investment landscapes.

### Core Drivers of Growth

The primary driver behind this boom is the exponential rise in AI training and inference workloads. Large language models (LLMs) and generative AI systems require massive computational power to process vast datasets and generate responses in real-time


## Generation Rollout works just fine
---